In [1]:
import os
import torch
import numpy as np
import librosa
from torch.utils.data import Dataset

class AudioDataset(Dataset):

    def __init__(self, root_dir, sr=16000, augment=False):

        self.files = []
        self.labels = []
        self.sr = sr
        self.augment = augment

        for label, folder in enumerate(sorted(os.listdir(root_dir))):
            folder_path = os.path.join(root_dir, folder)

            for file in os.listdir(folder_path):
                self.files.append(os.path.join(folder_path, file))
                self.labels.append(label)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        try:
            audio, _ = librosa.load(self.files[idx], sr=self.sr)
        except:
            return self.__getitem__((idx + 1) % len(self.files))

        max_length = self.sr * 10
        if self.augment:
            shift = np.random.randint(self.sr)
            audio = np.roll(audio, shift)

            noise = np.random.randn(len(audio))
            audio = audio + 0.003 * noise

            steps = np.random.uniform(-2, 2)
            audio = librosa.effects.pitch_shift(audio, sr=self.sr, n_steps=steps)

            rate = np.random.uniform(0.9, 1.1)
            audio = librosa.effects.time_stretch(audio, rate=rate)

        if len(audio) < max_length:
            audio = np.pad(audio, (0, max_length - len(audio)))
        else:
            audio = audio[:max_length]

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=self.sr,
            n_mels=128
        )

        mel_db = librosa.power_to_db(mel)

        if self.augment:

            t = mel_db.shape[1]
            t_mask = np.random.randint(10, 30)
            t0 = np.random.randint(0, t - t_mask)
            mel_db[:, t0:t0+t_mask] = mel_db.min()

            f = mel_db.shape[0]
            f_mask = np.random.randint(5, 20)
            f0 = np.random.randint(0, f - f_mask)
            mel_db[f0:f0+f_mask, :] = mel_db.min()

        mel_db = torch.tensor(mel_db).unsqueeze(0).float()
        
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        
        return mel_db, self.labels[idx]

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [3]:
from torch.utils.data import DataLoader
dataset = AudioDataset("/kaggle/input/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/Data/genres_original",augment=True)
loader = DataLoader(dataset,batch_size=32,shuffle=False)

In [4]:
x,y = dataset[0]
print(x.shape)

torch.Size([1, 128, 313])


In [5]:
import torch
import torch.nn as nn

class CnnModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(1, 16, 3),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1),

            nn.Conv2d(16, 32, 3),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1),

            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            # Block 2
            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),

            nn.Conv2d(64, 128, 3),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),

            nn.MaxPool2d(2),
            nn.Dropout(0.3),

            # Block 3
            nn.Conv2d(128, 128, 3),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),

            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.4),
            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.pool(x)
        #flatten after conv layers
        x = torch.flatten(x,1)
        x = self.classifier(x)

        return x

In [6]:
import torch.nn as nn
import torch.optim as optim

model = CnnModel().to(device)  ##send my model to GPU

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

In [7]:
print(device)

epochs = 4
patience = 2

best_loss = float("inf")
patience_counter = 0

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for audio, label in loader:

        audio = audio.to(device)
        label = label.to(device)

        output = model(audio)

        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(loader)

    scheduler.step(train_loss)

    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"LR: {current_lr}")

    if train_loss < best_loss:
        best_loss = train_loss
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered")
        break

cuda


/tmp/ipykernel_24/1376729537.py:29: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(self.files[idx], sr=self.sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 1
Train Loss: 2.4771
LR: 0.0003


/tmp/ipykernel_24/1376729537.py:29: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(self.files[idx], sr=self.sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 2
Train Loss: 2.3544
LR: 0.0003


/tmp/ipykernel_24/1376729537.py:29: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(self.files[idx], sr=self.sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 3
Train Loss: 2.3354
LR: 0.0003


/tmp/ipykernel_24/1376729537.py:29: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(self.files[idx], sr=self.sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 4
Train Loss: 2.3306
LR: 0.0003
